# LLaMA-2

[llama2](pics/llama2.png)

---

## 一、Rotary Positional Encoding

假设$d$维嵌入向量为$x$, token位置为$m$, 旋转矩阵为$\mathbf{R}_{\theta, m}^d$, 则经过旋转位置编码的向量$\mathbf{R}_{\theta, m}^d x$为：
$$
\mathbf{R}_{\theta, m}^d x = 
\begin{bmatrix}
x_1 \\ x_2 \\ x_3 \\ x_4 \\ \dots \\ x_{d-1} \\ x_d
\end{bmatrix} \otimes 
\begin{bmatrix}
\cos(m\theta_1) \\ \cos(m\theta_1) \\ \cos(m\theta_2) \\ \cos(m\theta_2) \\ \dots \\ \cos(m\theta_{d/2}) \\ \cos(m\theta_{d/2})
\end{bmatrix} +
\begin{bmatrix}
-x_2 \\ x_1 \\ -x_4 \\ x_3 \\ \dots \\ -x_{d} \\ x_{d-1}
\end{bmatrix} \otimes
\begin{bmatrix}
\sin(m\theta_1) \\ \sin(m\theta_1) \\ \sin(m\theta_2) \\ \sin(m\theta_2) \\ \dots \\ \sin(m\theta_{d/2}) \\ \sin(m\theta_{d/2})
\end{bmatrix}
$$
其中$\theta_i = 10000^{-2(i-1)/d}$, $d$维数被划分为相邻两两一组。这种计算方法**比直接进行旋转矩阵乘法快得多**。

In [1]:
import torch
import torch.nn as nn

"""
旋转位置编码: 一种相对位置编码，通过对原始向量进行旋转操作得到位置编码
对于一个query向量Wq*x, 通过对其乘以一个**旋转矩阵R**得到对应的旋转位置编码 
"""
def precompute_theta_pos_frequencies(dim: int, seq_len: int, device: str):
    assert dim % 2 == 0, "dimension must be even"
    
    """
    theta.shape: (dim/2)
    """
    i_iter = torch.arange(0, dim, 2).float()
    theta = 1.0 / (10000 ** (i_iter / dim)).to(device)
    
    """
    m.shape(position): (seq_len)
    freqs: (seq_len) @ (dim/2) -> (seq_len, dim/2), 即每个位置m都和所有的theta组合相乘, 得到正余弦内的数值
    """
    m = torch.arange(seq_len).to(device)
    freqs = torch.outer(m, theta).float()
    
    """
    torch.polar: 构建一个复数张量, 其元素模长均为1, 其元素角度来自freqs, 即复数为cos(freq)+i*sin(freq)
    freqs_complex.shape: (seq_len, dim/2)
    """
    freqs_complex = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_complex

那么如何将一个输入嵌入向量$x$按公式(1)转换为带有位置编码信息的旋转向量呢？假设$d = 4$。

1. 借助复数向量, 将输入$x$的维度**相邻两两组合**, 维度变成dim/2 (与freq的维度保持一致)：
$$
\begin{bmatrix}
x_1 \\ x_2 \\ x_3 \\ x_4
\end{bmatrix} \Rightarrow \begin{bmatrix}
x_1 + i x_2 \\ x_3 + i x_4
\end{bmatrix}
$$
- 另外由上面的freq复数化得到的复数张量为:
$$
\begin{bmatrix}
\cos(m\theta_1) + i \sin(m\theta_1) \\
\cos(m\theta_2) + i \sin(m\theta_2) \\
\end{bmatrix}
$$

2. 将上述两个张量进行**逐元素相乘**, 得到旋转后的复数矩阵:
$$
\begin{bmatrix}
x_1 \cos(m\theta_1) - x_2 \sin(m\theta_1) + i [x_2 \sin(m\theta_1) + x_1\cos(m\theta_1)] \\
x_3 \cos(m\theta_2) - x_4 \sin(m\theta_2) + i [x_4 \sin(m\theta_2) + x_3\cos(m\theta_2)] \\
\end{bmatrix}
$$

3. 复向量**实数化**, 并拉直:
$$
\begin{bmatrix}
x_1 \cos(m\theta_1) - x_2 \sin(m\theta_1) \\
x_2 \cos(m\theta_1) + x_1 \sin(m\theta_1) \\
x_3 \cos(m\theta_2) - x_4 \sin(m\theta_2) \\
x_4 \cos(m\theta_2) + x_3 \sin(m\theta_2) \\
\end{bmatrix}
$$
- 可见上述公式与开头旋转编码定义的公式计算结果完全相同, 具体代码实现如下。

In [2]:
def apply_rotatary_embeddings(x: torch.Tensor, freqs_complex: torch.Tensor, device: str):
    """
    先将x的最后一维扩成2个 (对应一组实数+复数)
        (batch, seq_len, n_heads, dim) -> (batch, seq_len, n_heads, dim/2, 2)
    再将x转化为复数, 与freqs_complex形状相同
        (batch, seq_len, n_heads, dim/2, 2) -> (batch, seq_len, n_heads, dim/2)
    """
    x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    
    """
    先扩展出批次和多头的维度, 便于与x_complex逐元素相乘
        (seq_len, dim/2) -> (1, seq_len, 1, dim/2)
    再与x_complex逐元素相乘
        (1, seq_len, 1, dim/2) * (batch, seq_len, n_heads, dim/2) -> (batch, seq_len, n_heads, dim/2)
    """
    freqs_complex = freqs_complex.unsqueeze(0).unsqueeze(2)
    x_rotated = x_complex * freqs_complex
    
    """
    先添加末尾2维度, 还原回实数张量
        (batch, seq_len, n_heads, dim/2) -> (batch, seq_len, n_heads, dim/2, 2)
    再拉直, 还原成输入x的形状 
        (batch, seq_len, n_heads, dim/2, 2) -> (batch, seq_len, n_heads, dim)
    """
    x_out = torch.view_as_real(x_rotated)
    x_out = x_out.reshape(*x.shape)
    
    return x_out.type_as(x).to(device)

---

## 二、RMS Normalization

RMSNorm是一种更加高效的样本归一化方法, 相较于层归一化方法其具有如下优势: 
1. 只需要计算一个统计量, 无需计算均值和方差
2. 不居中值, 而是缩小值, 效果更优

$$
\bar{a}_i = \dfrac{a_i}{\textbf{RMS}(a)} g_i, \qquad \text{where} \quad \textbf{RMS}(a) = \sqrt{\dfrac{1}{N} \sum_{i=1}^{N} a_i^2}
$$
其中 $g_i$ 是一个**可学习参数**。

In [3]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float=1e-6):
        super().__init__()
        self.eps = eps  # 防止除以0
        self.g = nn.Parameter(torch.ones(dim))
    
    def _norm(self, x: torch.Tensor):
        """
        rescale: (batch, seq_len, dim) / (batch, seq_len, 1) -> (batch, seq_len, dim)
        """
        return x / torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x: torch.Tensor):
        """
        (dim) * (batch, seq_len, dim) -> (batch, seq_len, dim)
        """
        return self.g * self._norm(x.float()).type_as(x)

---

## 三、Self-Attention

In [4]:
from dataclasses import dataclass
from typing import Optional

"""
dataclass: python装饰器, 定义一个类存储数据
自动生成__init__方法、__repr__等方法, 减少代码编写量
"""
@dataclass
class ModelArgs:
    dim: int = 4096
    num_layers: int = 32
    num_q_heads: int = 32
    num_kv_heads: Optional[int] = None # q, kv的在GQA中的多头数量可以不一样
    vocab_size: int = 30000
    multiple_of: int = 256
    ffn_dim_multiplier: Optional[float] = None
    norm_eps: float = 1e-5
    max_batch_size: int = 32
    max_seq_len: int = 2048
    device: str = None

In [5]:
class SelfAttention(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.num_kv_heads = args.num_q_heads if args.num_kv_heads is None else args.num_kv_heads
        self.num_q_heads = args.num_q_heads
        self.num_repeat = args.num_q_heads // args.num_kv_heads
        self.head_dim = args.dim // args.num_q_heads
        
        self.wq = nn.Linear(args.dim, args.num_q_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(args.dim, args.num_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(args.dim, args.num_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(args.num_q_heads * self.head_dim, args.dim, bias=False)
        
        self.cache_k = torch.zeros(size=(args.max_batch_size, args.max_seq_len, args.num_kv_heads, self.head_dim))
        self.cache_v = torch.zeros(size=(args.max_batch_size, args.max_seq_len, args.num_kv_heads, self.head_dim))
    
    def forward(self, x: torch.Tensor, start_pos: int, freqs_complex: torch.Tensor):
        # TO BE DONE
        return None
        